In [24]:
# !pip install numpy
# !pip install matplotlib
# !pip install pandas
# !pip install -U pyreadstat pandas
# !pip install scipy
# !pip install statsmodels

In [25]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import statsmodels.api as sm

# Resume Experiment Analysis

How much harder is it to get a job in the United States if you are Black than if you are White? Or, expressed differently, what is the *effect* of race on the difficulty of getting a job in the US?

In this exercise, we will be analyzing data from a real world experiment designed to help answer this question. Namely, we will be analyzing data from a randomized experiment in which 4,870 ficticious resumes were sent out to employers in response to job adverts in Boston and Chicago in 2001. The resumes differ in various attributes including the names of the applicants, and different resumes were randomly allocated to job openings. 

The "experiment" part of the experiment is that resumes were randomly assigned Black- or White-sounding names, and then watched to see whether employers called the "applicants" with Black-sounding names at the same rate as the applicants with the White-sounding names.

(Which names constituted "Black-sounding names" and "White-sounding names" was determined by analyzing names on Massachusetts birth certificates to determine which names were most associated with Black and White children, and then surveys were used to validate that the names were perceived as being associated with individuals of one racial category or the other. Also, please note I subscribe to the logic of [Kwame Anthony Appiah](https://www.theatlantic.com/ideas/archive/2020/06/time-to-capitalize-blackand-white/613159/) and chose to capitalize both the B in Black and the W in White). 

You can get access to original article [here](https://www.aeaweb.org/articles?id=10.1257/0002828042002561). 

**Note to Duke students:** if you are on the Duke campus network, you'll be able to access almost any academic journal articles directly; if you are off campus and want access, you can just go to the [Duke Library](https://library.duke.edu/) website and search for the article title. Once you find it, you'll be asked to log in, after which you'll have full access to the article. You will also find this pattern holds true at nearly any major University in the US.



## Gradescope Autograding

Please follow [all standard guidance](https://www.practicaldatascience.org/ids720_specific/autograder_guidelines.html) for submitting this assignment to the Gradescope autograder, including storing your solutions in a dictionary called `results` and ensuring your notebook runs from the start to completion without any errors.

For this assignment, please name your file `exercise_resume_experiment.ipynb` before uploading.

You can check that you have answers for all questions in your `results` dictionary with this code:

```python
assert set(results.keys()) == {
    "ex2_pvalue_computerskills",
    "ex2_pvalue_female",
    "ex2_pvalue_yearsexp",
    "ex3_pvalue_education",
    "ex4_validity",
    "ex5_pvalue",
    "ex5_white_advantage_percent",
    "ex5_white_advantage_percentage_points",
    "ex6_black_pvalue",
    "ex8_black_college",
    "ex8_black_nocollege",
    "ex8_college_heterogeneity",
    "ex9_gender_and_discrimination",
    "ex10_experiment_v_us",
}
```


### Submission Limits

Please remember that you are **only allowed THREE submissions to the autograder.** Your last submission (if you submit 3 or fewer times), or your third submission (if you submit more than 3 times) will determine your grade Submissions that error out will **not** count against this total.

## Checking for Balance

The first step in analyzing any experiment is to check whether you have *balance* across your treatment arms—that is to say, do the people who were randomly assigned to the treatment group look like the people who were randomly assigned to the control group. Or in this case, do the resumes that ended up with Black-sounding names look like the resumes with White-sounding names. 

Checking for balance is critical for two reasons. First, it's always possible that random assignment will create profoundly different groups—the *Large of Large Numbers* is only a "law" in the limit. So we want to make sure we have reasonably similar groups from the outset. And second, it's also always possible that the randomization wasn't actually implemented correctly—you would be amazed at the number of ways that "random assignment" can go wrong! So if you ever do find you're getting unbalanced data, you should worry not only about whether the groups have baseline differences, but also whether the "random assignment" was actually random!

### Exercise 1

Download the data set from this experiment (`resume_experiment.dta`) from [github](https://github.com/nickeubank/MIDS_Data/tree/master/resume_experiment). To aid the autograder, please load the data directly from a URL.


In [26]:
# import file resume_experiement.data dataset
# load directly from github url

data_url = "https://github.com/nickeubank/MIDS_Data/raw/refs/heads/master/resume_experiment/resume_experiment.dta"
df = pd.read_stata(data_url)
df.head()

,education,ofjobs,yearsexp,computerskills,call,female,black
0,4,2,6,1,0.0,1.0,0.0
1,3,3,6,1,0.0,1.0,0.0
2,4,1,6,1,0.0,1.0,1.0
3,3,4,6,1,0.0,1.0,1.0
4,3,3,22,1,0.0,1.0,0.0



### Exercise 2

- `black` is the treatment variable in the data set (whether the resume has a "Black-sounding" name).
- `call` is the dependent variable of interest (did the employer call the fictitious applicant for an interview)

In addition, the data include a number of variables to describe the other features in each fictitious resume, including applicants education level (`education`), years of experience (`yearsexp`), gender (`female`), computer skills (`computerskills`), and number of previous jobs (`ofjobs`). Each resume has a random selection of these attributes, so on average the Black-named fictitious applicant resumes have the same qualifications as the White-named applicant resumes. 

Check for balance in terms of the average values of applicant gender (`female`), computer skills (`computerskills`), and years of experience (`yearsexp`) across the two arms of the experiment (i.e. by `black`). Calculate both the differences in means across treatment arms *and* test for statistical significance of these differences. Does gender, computer skills, and yearsexp look balanced across race groups in terms of both statistical significance and magnitude of difference?

Store the p-values associated with your t-test of these variables in `ex2_pvalue_female`, `ex2_pvalue_computerskills`, and `ex2_pvalue_yearsexp`. **Round your values to 2 decimal places.**


In [27]:
# t test for difference in means of computer skill across black and white applicants


def t_test_by_group(dataframe, test_cols):
    results = {}
    for col in test_cols:
        group_black = dataframe[dataframe["black"] == 1.0][col]
        group_white = dataframe[dataframe["black"] == 0.0][col]

        mean_diff = group_white.mean() - group_black.mean()
        t_statistic, p_value = stats.ttest_ind(group_white, group_black)

        results[f"ex2_pvalue_{col}"] = round(p_value, 2)

        print("\n**********************************************)))")
        print(f"Column: {col}")
        print(f"Mean Difference (white - black): {mean_diff}")
        print(f"T-statistic: {t_statistic}")
        print(f"P-value: {p_value}\n")

    return results


test_columns = ["computerskills", "female", "yearsexp"]
results = t_test_by_group(df, test_columns)
results


**********************************************)))
Column: computerskills
Mean Difference (white - black): -0.023819301848049257
T-statistic: -2.1664271042751966
P-value: 0.030326933955391957


**********************************************)))
Column: female
Mean Difference (white - black): -0.010677635669708252
T-statistic: -0.8841321468353271
P-value: 0.3766682744026184


**********************************************)))
Column: yearsexp
Mean Difference (white - black): 0.026694045174538772
T-statistic: 0.18461970685747395
P-value: 0.8535350182481284



{'ex2_pvalue_computerskills': np.float64(0.03),
 'ex2_pvalue_female': np.float32(0.38),
 'ex2_pvalue_yearsexp': np.float64(0.85)}

**Interpretation:** Based on the t-test results above, applicant gender and years of experience are well balanced across Black- and White-named resumes in both statistical and practical terms, as the p values are not significant enough to reject the null hypothesis and the mean differences between the black and white groups are miniscule. Computer skills show a statistically significant difference, but the magnitude of the difference is very small, suggesting no meaningful imbalance. In general, the randomization seems successful.

### Exercise 3

Do a similar tabulation for education (`education`). Education is a categorical variable coded as follows:

- 0: Education not reported
- 1: High school dropout
- 2: High school graduate
- 3: Some college
- 4: College graduate or higher

Because these are categorical, you shouldn't just calculate and compare means—you should compare share or count of observations with each value (e.g., a chi-squared contingency table). You may also find the `pd.crosstab` function useful.

Does education look balanced across racial groups?

Store the p-value from your chi squared test in results under the key `ex3_pvalue_education`. **Please round to 2 decimal places.**

In [28]:
contingency_table_education = pd.crosstab(df["education"], df["black"])
contingency_table_education

black,0.0,1.0
education,,
0,18,28
1,18,22
2,142,132
3,513,493
4,1744,1760


In [29]:
# output table
chi2_statistic_education, p_value_education, dof_education, expected_education = (
    stats.chi2_contingency(contingency_table_education)
)
print(f"Chi-squared statistic for education: {chi2_statistic_education}")
print(f"P-value for education: {p_value_education}")
results["ex3_pvalue_education"] = round(p_value_education, 2)

Chi-squared statistic for education: 3.4095502219737974
P-value for education: 0.4917640058792273


**Interpretation:** Education appears well balanced across Black-named and White-named resumes, as the chi-squared test shows no statistically significant difference (p value of 0.49) in the distribution of education levels across the two race groups. The continegency table contains similar valued counts for each education group across the black and white groups. 

### Exercise 4

What do you make of the overall results on resume characteristics? Why do we care about whether these variables look similar across the race groups? And if they didn't look similar, would that be a threat to internal or external validity? 

Answer in markdown, then also store your answer to the question of whether imbalances are a threat to internal or external validity in `"ex4_validity"` as the string `"internal"` or `"external"`.


**Interpretation**: Overall, the resume characteristics appear well balanced across Black-named and White-named resumes. Most variables show no statistically significant differences (except for Computer skills), and any small differences that do appear are minor in magnitude. This suggests that the randomization worked as hoped and that later, differences in callback rates can be attributed to the race signal rather than underlying differences in applicant qualifications.

We care about balance because similar observable characteristics across race groups help isolate the causal effect of race on employer callbacks. If these variables were not balanced, it would threaten **internal validity**, since 
resume charisteristics could be affecting the differences in outcomes rather than just race.

In [30]:
ex4_validity = "internal"
results["ex4_validity"] = ex4_validity

## Estimating Effect of Race

### Exercise 5

The variable of interest in the data set is the variable `call`, which indicates a call back for an interview. Perform a two-sample t-test comparing applicants with black sounding names and white sounding names.

Interpret your results—in both percentage *and* in percentage points, what is the effect of having a Black-sounding name (as opposed to a White-sounding name) on your resume?

Store how much more likely a White applicant is to receive a call back than a Black respondent in percentage and percentage points in `"ex5_white_advantage_percent"`and `"ex5_white_advantage_percentage_points"`. Please scale percentages so 1 is 1% and percentage points so a value of `1` corresponds to 1 percentage point. **Please round these answers to 2 decimal places.**

Store the p-value of the difference in `"ex5_pvalue"` **Please round your p-value to 5 decimal places.**

In [31]:
# two sample t test on variable call comparing black and white applicants
group_black = df[df["black"] == 1.0]["call"]
group_white = df[df["black"] == 0.0]["call"]

ex5_tstat, ex5_pvalue = stats.ttest_ind(group_white, group_black, equal_var=False)

# means
mean_black = group_black.mean()
mean_white = group_white.mean()

# percentage points
ex5_white_advantage_percentage_points = round((mean_white - mean_black) * 100, 2)

# percentage
ex5_white_advantage_percent = round(((mean_white - mean_black) / mean_black) * 100, 2)

ex5_pvalue = round(ex5_pvalue, 5)

results["ex5_pvalue"] = ex5_pvalue
results["ex5_white_advantage_percent"] = ex5_white_advantage_percent
results["ex5_white_advantage_percentage_points"] = ex5_white_advantage_percentage_points

print(f"P-value for difference in call rates: {ex5_pvalue}")
print(
    f"White advantage in call rates: {ex5_white_advantage_percent}% "
    f"({ex5_white_advantage_percentage_points} percentage points)"
)

P-value for difference in call rates: 3.9999998989515007e-05
White advantage in call rates: 49.68000030517578% (3.200000047683716 percentage points)


### Exercise 6

Now, use a linear probability model (a linear regression with a 0/1 dependent variable!) to estimate the differential likelihood of being called back by applicant race (i.e. the racial discrimination by employers). Please use [statsmodels](https://www.statsmodels.org/stable/index.html).

Since we have a limited dependent variable, be sure to use [heteroskedastic robust standard errors.](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLSResults.get_robustcov_results.html) Personally, I prefer the `HC3` implementation, as it tends to do better with smaller samples than other implementations.

Interpret these results—what is the *effect* of having a Black-sounding name (as opposed to a White-sounding name) on your resume in terms of the likelihood you'll be called back? 

How does this compare to the estimate you got above in exercise 5?

Store the p-value associated with `black` in `"ex6_black_pvalue"`. **Please round your pvalue to 5 decimal places.**

In [32]:
X = df[["black"]]
X = sm.add_constant(X)
y = df["call"]

lpm = sm.OLS(y, X).fit(cov_type="HC3")

print(lpm.summary())
ex6_black_pvalue = round(lpm.pvalues["black"], 5)
results["ex6_black_pvalue"] = ex6_black_pvalue

print(f"P-value for difference in call rates: {ex6_black_pvalue}")

                            OLS Regression Results                            
Dep. Variable:                   call   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     16.92
Date:                Mon, 26 Jan 2026   Prob (F-statistic):           3.96e-05
Time:                        19:44:13   Log-Likelihood:                -562.24
No. Observations:                4870   AIC:                             1128.
Df Residuals:                    4868   BIC:                             1141.
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0965      0.006     16.121      0.0

**Interpretation:**

Exercise 5 -- P-value for difference in call rates: 3.9999998989515007e-05

Exercise 6 -- P-value for difference in call rates: 4e-05

From exercise 6, we see that gaving a Black-sounding name lowers the probability of receiving a callback by about 3.2 percentage points compared to a White-sounding name, and this effect is highly statistically significant using HC3 robust standard errors (p of 4e-05). This p value estimate is nearly identical to our result from exercise 5, as both the t-test and the linear probability model capture the same difference in mean callback rates.



### Exercise 7

Even when doing a randomized experiment, adding control variables to your regression *can* improve the statistical efficiency of your estimates of the treatment effect (the upside is the potential to explain residual variation; the downside is more parameters to be estimated). Adding controls can be particularly useful when randomization left some imbalances in covariates (which you may have seen above). 

Now let's see if we can improve our estimates by adding in other variables as controls. Add in `education`, `yearsexp`, `female`, and `computerskills`—be sure to treat education as a categorical variable!

In [33]:
# treat education as categorical variable
df["education_cat"] = df["education"].astype("category")
X = df[["black", "education_cat", "yearsexp", "female", "computerskills"]]
X = sm.add_constant(X)
y = df["call"]

lpm_with_controls = sm.OLS(y, X).fit(cov_type="HC3")

print(lpm_with_controls.summary())
ex7_black_pvalue = round(lpm_with_controls.pvalues["black"], 5)

print(f"P-value for difference in call rates with controls: {ex7_black_pvalue}")

                            OLS Regression Results                            
Dep. Variable:                   call   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     6.921
Date:                Mon, 26 Jan 2026   Prob (F-statistic):           1.90e-06
Time:                        19:44:13   Log-Likelihood:                -551.02
No. Observations:                4870   AIC:                             1114.
Df Residuals:                    4864   BIC:                             1153.
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0846      0.025      3.

**Interpretation:**

Even after we controlled for education, years of experience, gender, and computer skills, the resumes with Black-sounding names are still about 3 percentage points less likely to receive a callback, and the effect is still highly statistically significant (p value of 5e-05). The estimate barely changes from the earlier results, which suggests that the callback gap isn’t explained by the resume characteristics and is driven by race itself.

## Estimating Heterogeneous Effects

### Exercise 8

As you may recall from some past readings (such as this one on the [migraine medication Aimovig](https://ds4humans.com/30_questions/15_answering_exploratory_questions.html#faithful-representations)), our focus on estimating *Average Treatment Effects* runs the risk of papering over variation in how individuals respond. In the case of Aimovig, for example, nearly no patients actually experienced the Average Treatment Effect of the medication; around half of patients experienced no benefit, while the other half experienced a benefit of about twice the average treatment effect.

So far in this analysis we've been focusing on the *average* effect of having a Black-sounding name (as compared to a White-sounding name). But we can actually use our regression framework to look for evidence of *heterogeneous treatment effects*—effects that are different for different types of people in our data. We accomplish this by *interacting* a variable we think may be related to experiencing a differential treatment effect with our treatment variable. For example, if we think that applicants with Black-sounding names who have a college degree are likely to experience less discrimination, we can interact `black` with an indicator for having a college degree. If having a college degree reduces discrimination, we could expect the interaction term to be positive. 

Is there more or less racial discrimination (the absolute magnitude difference in call back rates between Black and White applicants) among applicants who have a college degree? Store your answer as the string `"more discrimination"` or `"less discrimination"` under the key `"ex8_college_heterogeneity"`.

Please still include `education`, `yearsexp`, `female`, and `computerskills` as controls.

**Note:** it's relatively safe to assume that someone hiring employees who sees a resume that does *not* report education levels will assume the applicant does not have a college degree. So treat "No education reported" as "not having a college degree."

In percentage points, what is the difference in call back rates:

- between White applicants without a college degree and Black applicants without a college degree (`ex8_black_nocollege`).
- between White applicants with a college degree and Black applicants with a college degree (`ex8_black_college`).

Use negative values to denote a lower probability for Black applicants to get a call back. **Scale so a value of `1` is a one percentage point difference. Please round your answer to 2 percentage points.**

Focus on the coefficient values, even if the significance is low.

In [34]:
df["college"] = (df["education"] == 4).astype(int)

# interaction term
df["black_college"] = df["black"] * df["college"]

X = df[
    [
        "black",
        "college",
        "black_college",
        "yearsexp",
        "female",
        "computerskills",
        "education_cat",
    ]
]


X = sm.add_constant(X)
y = df["call"]

lpm_q8 = sm.OLS(y, X).fit(cov_type="HC3")
print(lpm_q8.summary())

                            OLS Regression Results                            
Dep. Variable:                   call   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     5.064
Date:                Mon, 26 Jan 2026   Prob (F-statistic):           9.67e-06
Time:                        19:44:13   Log-Likelihood:                -550.76
No. Observations:                4870   AIC:                             1118.
Df Residuals:                    4862   BIC:                             1169.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0883      0.030      2.

In [35]:
beta_black = lpm_q8.params["black"]
beta_inter = lpm_q8.params["black_college"]

ex8_black_nocollege = round(beta_black * 100, 2)
ex8_black_college = round((beta_black + beta_inter) * 100, 2)

if abs(ex8_black_college) < abs(ex8_black_nocollege):
    ex8_college_heterogeneity = "less discrimination"
else:
    ex8_college_heterogeneity = "more discrimination"

print(
    f"Black applicants without college degree face {ex8_black_nocollege} percentage-point lower call rates."
)
print(
    f"Black applicants with college degree face {ex8_black_college} percentage-point lower call rates."
)
print(f"College degree leads to {ex8_college_heterogeneity} for black applicants.")
results["ex8_black_nocollege"] = ex8_black_nocollege
results["ex8_black_college"] = ex8_black_college
results["ex8_college_heterogeneity"] = ex8_college_heterogeneity

Black applicants without college degree face -4.05 percentage-point lower call rates.
Black applicants with college degree face -2.82 percentage-point lower call rates.
College degree leads to less discrimination for black applicants.


**Interpretation:**

Black applicants without a college degree are about 4.05 percentage points less likely to receive a callback than comparable White applicants, while Black applicants with a college degree are about 2.82 percentage points less likely to receive a callback. Because the absolute gap is smaller among college graduates, this indicates less discrimination for applicants with a college degree.

### Exercise 9

Now let's compare men and women—is the penalty for having a Black-sounding name greater for Black men or Black women? Store your answer as `"greater discrimination for men"` or `"greater discrimination for women"` in `"ex9_gender_and_discrimination"`.

Focus on the coefficient values, even if the significance is low.

Again, please still include `education`, `yearsexp`, `female`, and `computerskills` as controls.

In [36]:
# Interaction term
df["black_female"] = df["black"] * df["female"]

X = df[
    [
        "black",
        "female",
        "black_female",
        "yearsexp",
        "computerskills",
        "education_cat",
    ]
]


X = sm.add_constant(X)
y = df["call"]

lpm_q9 = sm.OLS(y, X).fit(cov_type="HC3")
print(lpm_q9.summary())

                            OLS Regression Results                            
Dep. Variable:                   call   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.007
Method:                 Least Squares   F-statistic:                     5.767
Date:                Mon, 26 Jan 2026   Prob (F-statistic):           5.41e-06
Time:                        19:44:13   Log-Likelihood:                -551.00
No. Observations:                4870   AIC:                             1116.
Df Residuals:                    4863   BIC:                             1161.
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0831      0.026      3.

In [37]:
beta_black = lpm_q9.params["black"]
beta_inter = lpm_q9.params["black_female"]

ex9_black_men = beta_black
ex9_black_women = beta_black + beta_inter

ex9_gender_and_discrimination = ""
if abs(ex9_black_men) > abs(ex9_black_women):
    ex9_gender_and_discrimination = "greater discrimination for men"
else:
    ex9_gender_and_discrimination = "greater discrimination for women"

print(f"Black men face {ex9_black_men} percentage-point lower call rates.")
print(f"Black women face {ex9_black_women} percentage-point lower call rates.")
print(f"Gender leads to {ex9_gender_and_discrimination} for black applicants.")

results["ex9_gender_and_discrimination"] = ex9_gender_and_discrimination

Black men face -0.028707826037032206 percentage-point lower call rates.
Black women face -0.032512421204455744 percentage-point lower call rates.
Gender leads to greater discrimination for women for black applicants.


**Interpretation:**

Black women face slightly more callback penalty from having a Black-sounding name than Black men, with about a 3.3 percentage-point reduction in callbacks compared to about 2.9 percentage points for Black men. This suggests that racial discrimination in callbacks is greater for women than for men in this sample, even though the difference between the two groups effects is still relatively small.

### Exercise 10

Calculate and/or lookup the following online:

- What is the share of applicants in our dataset with college degrees?
- What share of Black adult Americans have college degrees (i.e. have completed a bachelors degree)?

Is the share of Black applicants with college degrees in this data `"greater"`, or `"less"` than in the US? Store your answer as one of those strings in `"ex10_experiment_v_us"`

In [38]:
# prop of all applicants with a college degree
prop_college_all = (df["education"] == 4).mean()
print(
    "Share of all applicants with a college degree:",
    round(prop_college_all * 100, 2),
    "%",
)

# prop of Black applicants with a college degree
prop_college_black = (df.loc[df["black"] == 1, "education"] == 4).mean()
print(
    "Share of Black applicants with a college degree:",
    round(prop_college_black * 100, 2),
    "%",
)

# external statistic:  ~26–30% of Black adults in the U.S. have college degrees
# compare Black applicants in the dataset to back adults in the U.S.
if prop_college_black > 0.30:
    ex10_experiment_v_us = "greater"
else:
    ex10_experiment_v_us = "less"

results["ex10_experiment_v_us"] = ex10_experiment_v_us
print(
    "Black applicants in the experiment have",
    ex10_experiment_v_us,
    "college attainment than Black adults in the U.S.",
)

Share of all applicants with a college degree: 71.95 %
Share of Black applicants with a college degree: 72.28 %
Black applicants in the experiment have greater college attainment than Black adults in the U.S.


**Interpretation:** 

According to The Education Trust's recent research, only about 30% of Black adults have attained some form of college degree. Other sources online have also had similar numbers, ranging from 26-30%. 

Source: https://edtrust.org/wp-content/uploads/2014/09/Black-Degree-Attainment_FINAL.pdf


About 72% of applicants in the dataset have a college degree, and the share among Black applicants is very similar at about 72% as well. This is much higher than the share of Black adults in the U.S. with college degrees (30%) suggesting that the experiment focuses on a more highly educated group than the general Black population.



### Exercise 11

Bearing in mind your answers to Exercise 8 and to Exercise 10, how do you think the Average Treatment Effect you estimated in Exercises 5 and 6 might generalize to the experience of the average Black American (i.e., how do you think the ATE for the average Black American would compare to the ATE estimated from this experiment)?


**Answer:**

Because the Black applicants in the experiment are much more educated than the average Black American (72% comparedd to 30%), and discrimination appears to be lower for college-educated applicants, the estimated effect in this study probably underestimates how much discrimination the average Black American faces. This means the real impact of having a Black-sounding name in the real world is probably worse than what our experiment's results show.

### Exercise 12

What does your answer to Exercise 10 imply about the study's *internal* validity?

**Answer:**

Exercise 10 shows that just because that Black applicants in the experiment are more likely to have college degrees than Black adults in the U.S. does not affect the study’s internal validity, because the resumes were randomly assigned black or white-sounding names. Thus, the comparison between Black and White applicants is still reasonable within the experiment, even if the sample isn’t representative of the overall population.

### Exercise 13

What does your answer to Exercise 10 imply about the study's *external* validity?

**Answer:**

External validity is about how well the results of a study apply to people outside of the experiment.

Our answer to exercise 10 suggests the study has less external validity, because the Black applicants in this experiment are much more educated than the average Black American. As a result, the findings may not fully generalize to the larger Black population, especially to those with lower levels of education.

## What Did We Just Measure?

It's worth pausing for a moment to think about exactly what we've measured in this experiment. Was it the effect of race on hiring? Or the difference in the experience of the average White job applicant from the average Black job applicant?

Well... no. What we have measured in this experiment is **just** the effect of having a Black-sounding name (as opposed to a White-sounding name) on your resume on the likelihood of getting a followup call from someone hiring in Boston or Chicago given identical resumes. In that sense, what we've measured is a small *piece* of the difference in the experience of Black and White Americans when seeking employment. As anyone looking for a job knows, getting a call-back is obviously a crucial step in getting a job, so this difference—even if it's just one part of the overall difference—is remarkable.

In [39]:
results

{'ex2_pvalue_computerskills': np.float64(0.03),
 'ex2_pvalue_female': np.float32(0.38),
 'ex2_pvalue_yearsexp': np.float64(0.85),
 'ex3_pvalue_education': np.float64(0.49),
 'ex4_validity': 'internal',
 'ex5_pvalue': np.float32(4e-05),
 'ex5_white_advantage_percent': np.float32(49.68),
 'ex5_white_advantage_percentage_points': np.float32(3.2),
 'ex6_black_pvalue': np.float64(4e-05),
 'ex8_black_nocollege': np.float64(-4.05),
 'ex8_black_college': np.float64(-2.82),
 'ex8_college_heterogeneity': 'less discrimination',
 'ex9_gender_and_discrimination': 'greater discrimination for women',
 'ex10_experiment_v_us': 'greater'}

In [40]:
assert set(results.keys()) == {
    "ex2_pvalue_computerskills",
    "ex2_pvalue_female",
    "ex2_pvalue_yearsexp",
    "ex3_pvalue_education",
    "ex4_validity",
    "ex5_pvalue",
    "ex5_white_advantage_percent",
    "ex5_white_advantage_percentage_points",
    "ex6_black_pvalue",
    "ex8_black_college",
    "ex8_black_nocollege",
    "ex8_college_heterogeneity",
    "ex9_gender_and_discrimination",
    "ex10_experiment_v_us",
}